# SOV Extraction — Validation

Compare the cached Content Understanding output against ground truth side-by-side.

**Inputs**

- `reference/cu-output/<stem>.json` — produced by [01_extract_sov.ipynb](01_extract_sov.ipynb)
- `reference/expected-output/<stem>.json` — generated by `seed_data.py`

This notebook is **fully offline** — no Azure calls, no SDK. Just JSON in, DataFrames out.


## 1. Setup


In [ ]:
from __future__ import annotations

import json
import re
from pathlib import Path

import pandas as pd

NB_DIR    = Path.cwd() if "__file__" not in globals() else Path(__file__).parent
DEMO_ROOT = NB_DIR.parent if NB_DIR.name == "notebooks" else NB_DIR

CU_DIR       = DEMO_ROOT / "reference" / "cu-output"
EXPECTED_DIR = DEMO_ROOT / "reference" / "expected-output"

# Toggle: also surface enrichment-expected fields (fabricated by seed_data, not in source).
SHOW_ENRICHMENT_GAPS = False

ACCOUNT_FIELD_MAP = {
    "InsuredName": "insured_name", "DBA": "dba", "MailingAddress": "mailing_address",
    "EffectiveDate": "effective_date", "ExpirationDate": "expiration_date",
    "PrimaryOperations": "primary_operations", "NAICS": "naics", "Currency": "currency",
    "ValuationDate": "valuation_date", "TotalInsuredValue": "total_insured_value",
    "LocationCount": "location_count", "BrokerName": "broker_name",
    "BrokerContact": "broker_contact", "BrokerEmail": "broker_email",
    "BrokerPhone": "broker_phone", "PreparedBy": "prepared_by", "PreparedDate": "prepared_date",
}

LOCATION_FIELD_MAP = {
    "LocationNumber": "location_number", "BuildingNumber": "building_number",
    "Street": "street", "City": "city", "State": "state", "Zip": "zip",
    "ConstructionType": "construction_type", "Occupancy": "occupancy",
    "OperationsDescription": "operations_description", "YearBuilt": "year_built",
    "Stories": "stories", "SquareFootage": "square_footage", "UnitCount": "unit_count",
    "BuildingValue": "building_value", "BPPValue": "bpp_value", "BIEEValue": "bi_ee_value",
    "Sprinklered": "sprinklered", "ProtectionClass": "protection_class",
    "RoofYear": "roof_year", "FloodZone": "flood_zone",
    "DistanceToCoastMiles": "distance_to_coast_mi", "Notes": "notes",
}


def _unwrap(field):
    if not field:
        return None
    for key in ("valueString", "valueNumber", "valueDate",
                "valueInteger", "valueBoolean"):
        if key in field and field[key] is not None:
            return field[key]
    if "valueArray" in field:
        return field["valueArray"]
    if "valueObject" in field:
        return field["valueObject"]
    return None


def cu_account(payload: dict) -> dict:
    contents = (payload.get("result") or payload).get("contents", [])
    fields = contents[0].get("fields", {}) if contents else {}
    return {snake: _unwrap(fields.get(pas)) for pas, snake in ACCOUNT_FIELD_MAP.items()}


def cu_locations(payload: dict) -> list[dict]:
    contents = (payload.get("result") or payload).get("contents", [])
    fields = contents[0].get("fields", {}) if contents else {}
    arr = _unwrap(fields.get("Locations")) or []
    out = []
    for entry in arr:
        obj = entry.get("valueObject", {}) if isinstance(entry, dict) else {}
        out.append({snake: _unwrap(obj.get(pas)) for pas, snake in LOCATION_FIELD_MAP.items()})
    return out


def expected_stem(cu_stem: str) -> str:
    return cu_stem[:-4] if cu_stem.lower().endswith("_sov") else cu_stem


# ---------- Comparison helpers ----------

_TRUE = {"yes", "y", "true", "t", "1", "sprinklered"}
_FALSE = {"no", "n", "false", "f", "0", "unsprinklered"}
_PLACEHOLDERS = {"", "-", "--", "—", "–", "n/a", "na", "n.a.", "none", "null", "tbd", "."}

_TRANSLATE = str.maketrans({
    "—": "-", "–": "-", "−": "-", "\u2010": "-", "\u2011": "-",
    "\u201c": '"', "\u201d": '"',
    "\u2018": "'", "\u2019": "'",
    "\u00a0": " ",
})

_WS_RE = re.compile(r"\s+")
_FOOTNOTE_RE = re.compile(r"\s*\(\d+\)\s*")
_MARGIN_PREFIX_RE = re.compile(r"^margin note:\s*", re.I)
_TRUNC_SUFFIX_RE = re.compile(r"[.\s]*\u2026+\s*$")
_TRAIL_PUNCT_RE = re.compile(r"[.,;:\s]+$")


def _norm(v):
    """Coerce a value into a comparable form -- handles common type / case / unicode mismatches.
    Strips: trailing punctuation, footnote markers '(1)', 'MARGIN NOTE:' prefix, trailing '…'."""
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return None
    if isinstance(v, bool):
        return v
    if isinstance(v, (int, float)):
        return float(v)
    s = str(v).translate(_TRANSLATE).strip()
    s = _WS_RE.sub(" ", s)
    if not s:
        return None
    low = s.lower()
    low = _MARGIN_PREFIX_RE.sub("", low)
    low = _FOOTNOTE_RE.sub(" ", low).strip()
    low = _TRUNC_SUFFIX_RE.sub("", low).strip()
    low = _TRAIL_PUNCT_RE.sub("", low).strip()
    if low in _PLACEHOLDERS:
        return None
    if low in _TRUE:
        return True
    if low in _FALSE:
        return False
    try:
        return float(low.replace(",", ""))
    except ValueError:
        return low


def values_match(a, e) -> bool:
    return _norm(a) == _norm(e)


# ---------- Source-fields filtering ----------

def source_fields(expected_payload: dict) -> tuple[set[str], set[str], set[int], set[str]]:
    """Return (account_allowed, location_allowed, image_only_locs, image_only_fields).
    If `_source_fields` key is missing entirely, allow all known fields.
    Empty list `[]` means 'no fields are in source' (e.g. Cascade account block lives in email)."""
    sf = expected_payload.get("_source_fields") or {}
    acct = set(sf["account"]) if "account" in sf else set(ACCOUNT_FIELD_MAP.values())
    loc = set(sf["location"]) if "location" in sf else set(LOCATION_FIELD_MAP.values())
    img_locs = set(sf.get("image_only_locations") or [])
    img_fields = set(sf.get("image_only_fields") or [])
    return acct, loc, img_locs, img_fields


## 2. Discover what's available

Show which CU outputs have a matching expected-output file.


In [ ]:
cu_files = sorted(CU_DIR.glob("*.json"))

inventory = pd.DataFrame([
    {
        "cu_file":       p.name,
        "expected_file": f"{expected_stem(p.stem)}.json",
        "expected_exists": (EXPECTED_DIR / f"{expected_stem(p.stem)}.json").exists(),
    }
    for p in cu_files
])
inventory


,cu_file,expected_file,expected_exists
0,01_acme_SOV.json,01_acme.json,True
1,02_cascade_SOV.json,02_cascade.json,True
2,03_magnolia_SOV.json,03_magnolia.json,True
3,04_summit_SOV.json,04_summit.json,True
4,05_heartland_SOV.json,05_heartland.json,True
5,06_coastal_SOV.json,06_coastal.json,True


## 3. Pick a file to validate

Set `TARGET_STEM` to one of the rows above (without the `.json` extension).


In [ ]:
# Default to the first available CU file. Change to e.g. "03_magnolia_SOV" to inspect another.
TARGET_STEM = cu_files[0].stem if cu_files else None
print(f"TARGET_STEM = {TARGET_STEM!r}")

cu_path = CU_DIR / f"{TARGET_STEM}.json"
expected_path = EXPECTED_DIR / f"{expected_stem(TARGET_STEM)}.json"

cu_payload = json.loads(cu_path.read_text(encoding="utf-8"))
expected   = json.loads(expected_path.read_text(encoding="utf-8"))

print(f"CU output:        {cu_path.relative_to(DEMO_ROOT)}")
print(f"Expected output:  {expected_path.relative_to(DEMO_ROOT)}")


TARGET_STEM = '01_acme_SOV'
CU output:        reference\cu-output\01_acme_SOV.json
Expected output:  reference\expected-output\01_acme.json


## 4. Account-level side-by-side


In [ ]:
actual_acct = cu_account(cu_payload)
expected_acct = expected["account"]
acct_allowed, loc_allowed, img_locs, img_fields = source_fields(expected)

account_df = pd.DataFrame({
    "actual":   pd.Series(actual_acct),
    "expected": pd.Series({k: expected_acct.get(k) for k in actual_acct}),
})
account_df["in_source"] = [k in acct_allowed for k in account_df.index]
account_df["match"] = [values_match(a, e) for a, e in zip(account_df["actual"], account_df["expected"])]

# Default view: only fields actually in the source document.
if SHOW_ENRICHMENT_GAPS:
    account_df
else:
    account_df[account_df["in_source"]]


## 5. Locations side-by-side

One row per (location_number, field) with `actual`, `expected`, and a boolean `match` column. Filter with `locations_df.query("not match")` to focus on differences.


In [ ]:
actual_locs   = cu_locations(cu_payload)
expected_locs = expected.get("locations", [])

print(f"Locations: actual={len(actual_locs)}, expected={len(expected_locs)}")


def _key(loc: dict):
    n = loc.get("location_number")
    try:
        return int(str(n).strip())
    except (TypeError, ValueError):
        return n


a_by = {_key(l): l for l in actual_locs}
e_by = {_key(l): l for l in expected_locs}
all_keys = sorted(set(a_by) | set(e_by), key=lambda x: (x is None, str(x)))

rows = []
for k in all_keys:
    a, e = a_by.get(k, {}), e_by.get(k, {})
    is_image_only = k in img_locs
    for field in LOCATION_FIELD_MAP.values():
        av, ev = a.get(field), e.get(field)
        if av is None and ev is None:
            continue
        # Determine if this field is in source for THIS location.
        if is_image_only:
            in_source = field in img_fields
        else:
            in_source = field in loc_allowed
        rows.append({
            "loc#":      k,
            "field":     field,
            "actual":    av,
            "expected":  ev,
            "in_source": in_source,
            "match":     values_match(av, ev),
        })

locations_df = pd.DataFrame(rows)
if SHOW_ENRICHMENT_GAPS:
    locations_df
else:
    locations_df[locations_df["in_source"]]


Locations: actual=22, expected=22


### Mismatches only


In [ ]:
# Mismatches restricted to fields that are actually in the source document.
view = locations_df if SHOW_ENRICHMENT_GAPS else locations_df[locations_df["in_source"]]
view.query("not match")


,loc#,field,actual,expected,in_source,match


## 7. Cross-file summary (in-source only)

Total mismatches per file, restricted to fields actually present in each source document.


In [ ]:
summary_rows = []
for cu_path_i in sorted(CU_DIR.glob("*.json")):
    exp_path = EXPECTED_DIR / f"{expected_stem(cu_path_i.stem)}.json"
    if not exp_path.exists():
        continue
    cu_p = json.loads(cu_path_i.read_text(encoding="utf-8"))
    ex_p = json.loads(exp_path.read_text(encoding="utf-8"))
    a_acct, l_allow, i_locs, i_fields = source_fields(ex_p)

    # Account mismatches (in-source only)
    act_acct = cu_account(cu_p)
    exp_acct = ex_p.get("account", {})
    acct_mis = sum(
        1 for k in a_acct
        if not values_match(act_acct.get(k), exp_acct.get(k))
    )

    # Location mismatches (in-source only)
    act_locs = cu_locations(cu_p)
    exp_locs = ex_p.get("locations", [])
    a_map = {_key(l): l for l in act_locs}
    e_map = {_key(l): l for l in exp_locs}
    loc_mis = 0
    for k in set(a_map) | set(e_map):
        a, e = a_map.get(k, {}), e_map.get(k, {})
        allowed = i_fields if k in i_locs else l_allow
        for f in allowed:
            if not values_match(a.get(f), e.get(f)):
                loc_mis += 1

    summary_rows.append({
        "file": cu_path_i.stem,
        "loc_count_actual":   len(act_locs),
        "loc_count_expected": len(exp_locs),
        "account_mismatches":  acct_mis,
        "location_mismatches": loc_mis,
    })

summary_df = pd.DataFrame(summary_rows)
summary_df.loc["TOTAL"] = summary_df[["account_mismatches", "location_mismatches"]].sum(numeric_only=True)
summary_df


,file,loc_count_actual,loc_count_expected,account_mismatches,location_mismatches
0,01_acme_SOV,22.0,22.0,0.0,0.0
1,02_cascade_SOV,8.0,8.0,0.0,0.0
2,03_magnolia_SOV,15.0,15.0,0.0,0.0
3,04_summit_SOV,12.0,12.0,0.0,0.0
4,05_heartland_SOV,10.0,10.0,0.0,0.0
5,06_coastal_SOV,6.0,6.0,0.0,0.0
TOTAL,NaN,NaN,NaN,0.0,0.0
